
# Оцінка та вдосконалення моделі


<h2>Мета</h2>

Ознайомитись з методами оцінки та вдосконалення регресійних моделей. Після завершення цієї лабораторної роботи ви зможете:

* Розділяти дані на навчальну та тестову вибірки
* Використовувати перехресну перевірку для оцінки якості моделі
* Обирати оптимальну складність моделі для уникнення перенавчання
* Вдосконалювати моделі прогнозування за допомогою підбору параметрів

Приблизний час виконання: **60** хвилин

# <h2>Зміст</h2>

<div class="alert alert-block alert-info" style="margin-top: 20px">
<h3>Завдання для тренування</h3>
<ul>
    <li><a href="#ref1">Оцінка моделі: навчання та тестування </a></li>
    <li><a href="#ref2">Overfitting, Underfitting та вибір моделі </a></li>
    <li><a href="#ref3">Гребенева регресія </a></li>
    <li><a href="#ref4">Підбір параметрів моделі (пошук по сітці)</a></li>
</ul>

</div>

<hr>

Імпорт бібліотек та даних:

In [ ]:
#install specific version of libraries used in lab
#! mamba install pandas==1.3.3 -y
#! mamba install numpy=1.21.2 -y
#! mamba install sklearn=0.20.1 -y
#! mamba install ipywidgets=7.4.2 -y

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

DATA_PATH = 'data/clean_data.csv'

df = pd.read_csv(DATA_PATH, sep=';', encoding='cp1252')

df[['country_name', 'region']] =  df[['country_name', 'region']].astype('string')
df.dtypes


country_name          string[python]
region                string[python]
gdp_per_capita                 int64
population                     int64
co2_emission                 float64
area                         float64
population_density           float64
dtype: object

 Спочатку будемо використовувати тільки числові дані:


In [2]:
df=df._get_numeric_data()
df.head()

,gdp_per_capita,population,co2_emission,area,population_density
0,5617787463,34656032,9809.225,652860.0,53.083405
1,412498239,2876101,5716.853,28750.0,100.038296
2,3916881571,40606052,145400.217,2381740.0,17.048902
3,3698862203,77281,462.042,470.0,164.427660
4,3308700233,28813463,34763.160,1246700.0,23.111786


<div class="alert alert-success alertsuccess" style="margin-top: 20px">
<h2> Завдання  #1:</h2>

<p>Використовуйте функцію "train_test_split", щоб розділити набір даних таким чином, щоб 40% використовувалося для тестування. Встановіть параметр "random_state" рівним нулю. Вихідні дані функції мають бути такими: "x_train1", "x_test1", "y_train1" і "y_test1".</p>
</div>


Записую дані у два датафрейми. x_data – предиктори, y_data - відгуки.

In [8]:
# Напишіть ваш код нижче та натисніть Shift+Enter для виконання
y_data = df["co2_emission"]
x_data = df.drop(columns=["co2_emission"], axis=1)


,gdp_per_capita,population,area,population_density
0,5617787463,34656032,652860.0,53.083405
1,412498239,2876101,28750.0,100.038296
2,3916881571,40606052,2381740.0,17.048902
3,3698862203,77281,470.0,164.427660
4,3308700233,28813463,1246700.0,23.111786


In [9]:
x_data

,gdp_per_capita,population,area,population_density
0,5617787463,34656032,652860.0,53.083405
1,412498239,2876101,28750.0,100.038296
2,3916881571,40606052,2381740.0,17.048902
3,3698862203,77281,470.0,164.427660
4,3308700233,28813463,1246700.0,23.111786
...,...,...,...,...
200,3675459700,31568179,912050.0,34.612334
201,2170648054,92701100,330967.0,280.091671
202,990334774,27584213,527970.0,52.245796
203,1269573537,16591390,752610.0,22.045136


In [10]:
y_data

0        9809.225
1        5716.853
2      145400.217
3         462.042
4       34763.160
          ...    
200    185220.170
201    166910.839
202     22698.730
203      4503.076
204     12020.426
Name: co2_emission, Length: 205, dtype: float64

Розділяю дані на навчальні та тестові таким чином, щоб 40% використовувалося для тестування. 

In [11]:
from sklearn.model_selection import train_test_split

x_train1, x_test1, y_train1, y_test1 = train_test_split(x_data, y_data, test_size=0.4, random_state=0)
print("number of test samples :", x_test1.shape[0])
print("number of training samples:",x_train1.shape[0])

number of test samples : 82
number of training samples: 123


<div class="alert alert-success alertsuccess" style="margin-top: 20px">
<h2> Завдання  #2: </h2>
    
<p>Знайдіть R^2  на тестових даних, використовуючи 40% загального набору в якості тестових даних.
</p>
</div>


In [ ]:
# Напишіть ваш код нижче та натисніть Shift+Enter для виконання


<details><summary>Натисніть тут, щоб побачити підказку</summary>

```python
x_train1, x_test1, y_train1, y_test1 = train_test_split(x_data, y_data, test_size=0.4, random_state=0)
lre.fit(x_train1[['horsepower']],y_train1)
lre.score(x_test1[['horsepower']],y_test1)

```

</details>


Іноді у вас недостатньо даних тестування; тоді можна виконати перехресну перевірку. Розглянемо кілька методів, які можна використовувати для перехресної перевірки.

<h3>Перехресна перевірка (Cross-Validation Score)</h3>


Імпортуємо <b>model_selection</b> з модуля <b>cross_val_score</b>.


In [ ]:
from sklearn.model_selection import cross_val_score

Вводимо об’єкт, ознаку ('horsepower') і цільові дані (y_data). Параметр "cv" визначає кількість фолдів, в нашому випадку їх 4 (при бажанні можна задавати іншу кількість, зазвичай в межах 4-10).

In [ ]:
Rcross = cross_val_score(lre, x_data[['horsepower']], y_data, cv=4)

Оцінка за умовчанням R^2. Кожен елемент у масиві має середнє значення R^2 для фолду:

In [ ]:
Rcross

 Можемо обчислити середнє значення та стандартне відхилення нашої оцінки:


In [ ]:
print("The mean of the folds are", Rcross.mean(), "and the standard deviation is" , Rcross.std())

Можемо використовувати від’ємну квадратичну помилку як оцінку, встановивши для параметра "scoring" метрику "neg_mean_squared_error".

In [ ]:
-1 * cross_val_score(lre,x_data[['horsepower']], y_data,cv=4,scoring='neg_mean_squared_error')

<div class="alert alert-success alertsuccess" style="margin-top: 20px">
<h2> Завдання  #3: </h2>
    
<p>Обчисліть середнє R^2, використовуючи два фолди, а потім знайдіть середнє R^2 для другого фолду, використовуючи ознаку 'horsepower':
</p>
</div>


In [ ]:
# Напишіть ваш код нижче та натисніть Shift+Enter для виконання


<details><summary>Натисніть тут, щоб побачити підказку</summary>

```python
Rc=cross_val_score(lre,x_data[['horsepower']], y_data,cv=2)
Rc.mean()

```

</details>


Ви також можете використовувати функцію "cross_val_predict" для прогнозування результату. Функція розбиває дані на вказану кількість фолдів, причому один фолд для тестування, а інші фолди використовуються для навчання. Спочатку імпортуйте функцію:

In [ ]:
from sklearn.model_selection import cross_val_predict

Вводимо об’єкт, ознаку 'horsepower' і цільові дані <b>y_data</b>. Параметр "cv" визначає кількість фолдів. У цьому випадку їх 4. Можемо отримати результат:

In [ ]:
yhat = cross_val_predict(lre,x_data[['horsepower']], y_data,cv=4)
yhat[0:5]

<a name="ref2"></a>
## <h2>Overfitting, Underfitting та вибір моделі</h2>

<p>Виявилося, що тестові дані, які іноді називають "даними поза вибіркою", є набагато кращим показником ефективності вашої моделі в реальному світі. Однією з причин цього є перенавчання (overfitting).

Розглянемо кілька прикладів. Виявилося, що ці відмінності більш очевидні в множинній лінійній регресії та поліноміальній регресії, тому досліджуватимемо перенавчання на прикладі цих моделей.</p>

Давайте створимо кілька об’єктів лінійної регресії та навчимо моделі, використовуючи 'horsepower', 'curb-weight', 'engine-size' і 'highway-mpg' як ознаки-предиктори.

In [ ]:
lr = LinearRegression()
lr.fit(x_train[['horsepower', 'curb-weight', 'engine-size', 'highway-mpg']], y_train)

Прогнозування з використанням навчальних даних:


In [ ]:
yhat_train = lr.predict(x_train[['horsepower', 'curb-weight', 'engine-size', 'highway-mpg']])
yhat_train[0:5]

Прогнозування з використанням тестових даних:


In [ ]:
yhat_test = lr.predict(x_test[['horsepower', 'curb-weight', 'engine-size', 'highway-mpg']])
yhat_test[0:5]

Виконаємо деяку оцінку моделі, використовуючи навчальні та тестові дані окремо. Спочатку імпортуємо бібліотеку seaborn і matplotlib для побудови діаграм.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

Розглянемо розподіл прогнозованих значень для навчальних даних.

In [ ]:
Title = 'Distribution  Plot of  Predicted Value Using Training Data vs Training Data Distribution'
DistributionPlot(y_train, yhat_train, "Actual Values (Train)", "Predicted Values (Train)", Title)

Рисунок 1: Діаграма прогнозованих значень на основі навчальних даних порівняно з фактичними значеннями навчальних даних.


Поки що модель, здається, добре навчається з навчального набору даних. Але що відбувається, коли модель зустрічає нові дані з тестового набору даних? Коли модель генерує нові значення з тестових даних, бачимо, що розподіл прогнозованих значень сильно відрізняється від фактичних цільових значень.

In [ ]:
Title='Distribution  Plot of  Predicted Value Using Test Data vs Data Distribution of Test Data'
DistributionPlot(y_test,yhat_test,"Actual Values (Test)","Predicted Values (Test)",Title)

Рисунок 2: Діаграма прогнозованих значень на основі тестових даних порівняно з фактичними значеннями тестових даних.

<p>При порівнянні рисунків стає очевидним, що розподіл тестових даних на рис.1 набагато краще відповідає даним. Ця різниця на рис.2 помітна в діапазоні від 5000 до 15 000. Тут форма розподілу надзвичайно відрізняється. </p>
<p>Давайте подивимося, чи поліноміальна регресія також демонструє падіння точності передбачення під час аналізу тестового набору даних.</p>

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

<h4>Overfitting</h4>
<p>Перенавчання відбувається, коли модель відповідає шуму, але не відповідає базовому процесу. Таким чином, під час тестування вашої моделі за допомогою тестового набору ваша модель не працює так добре, оскільки вона моделює шум, а не основний процес, який породив зв’язок. </p>

Створимо поліноміальну модель 5 ступеня. Використаємо 55 відсотків даних для навчання, а решту для тестування:


In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x_data, y_data, test_size=0.45, random_state=0)

Виконаємо поліноміальне перетворення 5 ступеня для ознаки 'horsepower'.

In [ ]:
pr = PolynomialFeatures(degree=5)
x_train_pr = pr.fit_transform(x_train[['horsepower']])
x_test_pr = pr.fit_transform(x_test[['horsepower']])
pr

Тепер створимо модель лінійної регресії "poly" і навчимо її.

In [ ]:
poly = LinearRegression()
poly.fit(x_train_pr, y_train)

Можемо побачити результати моделі за допомогою методу "predict". Присвоюємо значення "yhat".

In [ ]:
yhat = poly.predict(x_test_pr)
yhat[0:5]

Давайте візьмемо перші п’ять прогнозованих значень і порівняємо їх із фактичними.


In [ ]:
print("Predicted values:", yhat[0:5])
print("True values:", y_test[0:5].values)

Будемо використовувати функцію "PollyPlot", яку ми визначили на початку лабораторної роботи, щоб відобразити навчальні дані,  тестові дані та прогнозовану функцію.

In [ ]:
PollyPlot(x_train[['horsepower']], x_test[['horsepower']], y_train, y_test, poly,pr)

Рисунок 3: Модель поліноміальної регресії, де червоні крапки представляють навчальні дані, зелені крапки представляють тестові дані, а синя лінія - це прогноз моделі.

Бачимо, що оціночна функція, здається, відстежує дані, але приблизно на 200 кінських силах функція починає відхилятися від точок даних.

 R^2 на навчальних даних:


In [ ]:
poly.score(x_train_pr, y_train)

 R^2 на тестових даних:


In [ ]:
poly.score(x_test_pr, y_test)

Бачимо, що R^2 для навчальних даних становить 0.5567, тоді як R^2 для тестових даних становив -29.87. Чим менше R^2, тим гірша модель. Від’ємний R^2 є ознакою перенавчання.

Давайте подивимося, як змінюється R^2 на тестових даних для поліномів різних порядків, а потім побудуємо результати:


In [ ]:
Rsqu_test = []

order = [1, 2, 3, 4]
for n in order:
    pr = PolynomialFeatures(degree=n)

    x_train_pr = pr.fit_transform(x_train[['horsepower']])

    x_test_pr = pr.fit_transform(x_test[['horsepower']])

    lr.fit(x_train_pr, y_train)

    Rsqu_test.append(lr.score(x_test_pr, y_test))

plt.plot(order, Rsqu_test)
plt.xlabel('order')
plt.ylabel('R^2')
plt.title('R^2 Using Test Data')
plt.text(3, 0.75, 'Maximum R^2 ')

Бачимо, що R^2 поступово збільшується, поки не буде використано поліном третього порядку. Далі R^2 різко зменшується для полінома четвертого порядку.


Наступний інтерфейс дозволяє експериментувати з різними порядками поліномів і різними обсягами даних.


In [ ]:
# Ця функція буде використана в наступному блоці. Будь ласка, запустіть на виконання
def f(order, test_data):
    x_train, x_test, y_train, y_test = train_test_split(x_data, y_data, test_size=test_data, random_state=0)
    pr = PolynomialFeatures(degree=order)
    x_train_pr = pr.fit_transform(x_train[['horsepower']])
    x_test_pr = pr.fit_transform(x_test[['horsepower']])
    poly = LinearRegression()
    poly.fit(x_train_pr,y_train)
    PollyPlot(x_train[['horsepower']], x_test[['horsepower']], y_train,y_test, poly, pr)

In [ ]:
interact(f, order=(0, 6, 1), test_data=(0.05, 0.95, 0.05))

<div class="alert alert-success alertsuccess" style="margin-top: 20px">
<h2> Завдання  #4a:</h2>

<p>Ми можемо виконувати поліноміальні перетворення з більш ніж однією ознакою. Створіть об'єкт "PolynomialFeatures" "pr1" другого ступеня.</p>
</div>


In [ ]:
# Напишіть ваш код нижче та натисніть Shift+Enter для виконання


<details><summary>Натисніть тут, щоб побачити підказку</summary>

```python
pr1=PolynomialFeatures(degree=2)

```

</details>


<div class="alert alert-success alertsuccess" style="margin-top: 20px">
<h2> Завдання  #4b: </h2>

<p>Перетворіть навчальну та тестову вибірки для ознак 'horsepower', 'curb-weight', 'engine-size' та 'highway-mpg'. </p><p>Підказка: використовуйте метод "fit_transform".</p>
</div>


In [ ]:
# Напишіть ваш код нижче та натисніть Shift+Enter для виконання


<details><summary>Натисніть тут, щоб побачити підказку</summary>

```python
x_train_pr1=pr1.fit_transform(x_train[['horsepower', 'curb-weight', 'engine-size', 'highway-mpg']])

x_test_pr1=pr1.fit_transform(x_test[['horsepower', 'curb-weight', 'engine-size', 'highway-mpg']])
```
</details>


<div class="alert alert-success alertsuccess" style="margin-top: 20px">
<h2> Завдання  #4c: </h2>
    
<p>Скільки вимірів має нова функція? </p><p>Підказка: використовуйте атрибут "shape".
</p>
</div>


In [ ]:
# Напишіть ваш код нижче та натисніть Shift+Enter для виконання


<details><summary>Натисніть тут, щоб побачити підказку</summary>

```python
x_train_pr1.shape #there are now 15 features
```

</details>


<div class="alert alert-success alertsuccess" style="margin-top: 20px">
<h2> Завдання  #4d: </h2>

<p>Створіть модель лінійної регресії "poly1". Навчіть об'єкт за допомогою методу "fit" використовуючи поліноміальні ознаки.</p></div>


In [ ]:
# Напишіть ваш код нижче та натисніть Shift+Enter для виконання


<details><summary>Натисніть тут, щоб побачити підказку</summary>

```python
poly1=LinearRegression().fit(x_train_pr1,y_train)
```

</details>


<div class="alert alert-success alertsuccess" style="margin-top: 20px">
<h2> Завдання  #4e: </h2>
    
<p>Використовуйте метод "predict", щоб спрогнозувати результати на поліноміальних ознаках, а потім скористайтеся функцією "DistributionPlot", щоб відобразити розподіл прогнозованих результатів для тестових даних порівняно з фактичними для тестових даних.</p>
</div>


In [ ]:
# Напишіть ваш код нижче та натисніть Shift+Enter для виконання


<details><summary>Натисніть тут, щоб побачити підказку</summary>

```python
yhat_test1=poly1.predict(x_test_pr1)

Title='Distribution  Plot of  Predicted Value Using Test Data vs Data Distribution of Test Data'

DistributionPlot(y_test, yhat_test1, "Actual Values (Test)", "Predicted Values (Test)", Title)
```
</details>


<div class="alert alert-success alertsuccess" style="margin-top: 20px">
<h2> Завдання  #4f: </h2>

<p>Використовуючи наведений вище графік розподілу, опишіть (словами) два регіони, де прогнозовані ціни менш точні, ніж фактичні ціни.</p>
</div>


In [ ]:
# Напишіть відповідь, виконувати не потрібно


<details><summary>Натисніть тут, щоб побачити підказку</summary>

```python
Прогнозована вартість вища за фактичну для автомобілів, ціна яких становить 10 000 доларів США,
навпаки, прогнозована ціна нижча за вартість у діапазоні від 30 000 до 40 000 доларів США.
Таким чином, модель не така точна в цих діапазонах.
```

</details>



<a name="ref3"></a>
## <h2>Гребенева регресія (Ridge Regression)</h2>


У цьому розділі розглянемо гребеневу регресію та побачимо, як параметр альфа змінює модель. Зверніть увагу: тут тестові дані використовуватимуться як дані для перевірки (validation data) правильності підбору параметра.

Виконаємо поліноміальне перетворення другого ступеня над нашими даними.


In [ ]:
pr=PolynomialFeatures(degree=2)
x_train_pr=pr.fit_transform(x_train[['horsepower', 'curb-weight', 'engine-size', 'highway-mpg','normalized-losses','symboling']])
x_test_pr=pr.fit_transform(x_test[['horsepower', 'curb-weight', 'engine-size', 'highway-mpg','normalized-losses','symboling']])

Імпортуємо <b>Ridge</b> з модуля <b>linear models</b>.

In [ ]:
from sklearn.linear_model import Ridge

Створимо об’єкт гребеневої регресії, встановивши параметр регуляризації (alpha) на 0.1

In [ ]:
RigeModel=Ridge(alpha=0.1)

Подібно до звичайної регресії будуємо модель за допомогою методу <b>fit</b>.


In [ ]:
RigeModel.fit(x_train_pr, y_train)

Отримуємо прогноз:


In [ ]:
yhat = RigeModel.predict(x_test_pr)

Порівняємо перші п’ять прогнозованих значень із нашим тестовим набором:


In [ ]:
print('predicted:', yhat[0:5])
print('test set :', y_test[0:5].values)

Далі вибираємо значення alpha, яке мінімізує помилку тесту. Для цього можемо використовувати цикл <code>for</code>. Ми також створили індикатор прогресу, щоб побачити, скільки ітерацій виконали на даний момент.

In [ ]:
from tqdm import tqdm

Rsqu_test = []
Rsqu_train = []
dummy1 = []
Alpha = 10 * np.array(range(0,1000))
pbar = tqdm(Alpha)

for alpha in pbar:
    RigeModel = Ridge(alpha=alpha)
    RigeModel.fit(x_train_pr, y_train)
    test_score, train_score = RigeModel.score(x_test_pr, y_test), RigeModel.score(x_train_pr, y_train)

    pbar.set_postfix({"Test Score": test_score, "Train Score": train_score})

    Rsqu_test.append(test_score)
    Rsqu_train.append(train_score)

Можемо побудувати значення R^2 для різних alpha:

In [ ]:
width = 12
height = 10
plt.figure(figsize=(width, height))

plt.plot(Alpha,Rsqu_test, label='validation data  ')
plt.plot(Alpha,Rsqu_train, 'r', label='training data ')
plt.xlabel('alpha')
plt.ylabel('R^2')
plt.legend()

Рисунок 4. Синя лінія позначає R^2 для даних перевірки (validation data), а червона лінія позначає R^2 для навчальних даних. Вісь х представляє різні значення Alpha.

Тут модель побудована та протестована на тих самих даних, тому навчальні та тестові дані однакові.

Червона лінія на рис.4 представляє R^2 для навчальних даних. Зі збільшенням alpha R^2 зменшується. Тому зі збільшенням alpha модель гірше працює з навчальними даними.

Синя лінія позначає R^2 для даних перевірки. Коли значення alpha збільшується, R^2 збільшується і сходиться в точці.

<div class="alert alert-success alertsuccess" style="margin-top: 20px">
<h2> Завдання  #5: </h2>

Побудуйте модель гребеневої регресії. Обчисліть R^2 за допомогою поліноміальних ознак, використовуйте навчальні дані для навчання моделі та використайте тестові дані для перевірки моделі. Параметр alpha повинен бути встановлений на 10.
</div>


In [ ]:
# Напишіть ваш код нижче та натисніть Shift+Enter для виконання


<details><summary>Натисніть тут, щоб побачити підказку</summary>

```python
RigeModel = Ridge(alpha=10)
RigeModel.fit(x_train_pr, y_train)
RigeModel.score(x_test_pr, y_test)

```

</details>



<a name="ref4"></a>
## <h2>Підбір параметрів моделі - пошук по сітці (Grid Search)</h2>


Термін alpha є гіперпараметром. Sklearn має клас <b>GridSearchCV</b>, щоб спростити процес пошуку найкращого значення гіперпараметра.

Імпортуємо <b>GridSearchCV</b> з модуля <b>model_selection</b>.


In [ ]:
from sklearn.model_selection import GridSearchCV

Створюємо словник значень параметрів:


In [ ]:
parameters1= [{'alpha': [0.001,0.1,1, 10, 100, 1000, 10000, 100000, 1000000]}]
parameters1

Створюємо об’єкт гребеневої регресії:


In [ ]:
RR=Ridge()
RR

Створюємо об’єкт сітки пошуку параметра гребеневої регресії:

In [ ]:
Grid1 = GridSearchCV(RR, parameters1, cv=4)

Підбираємо модель:

In [ ]:
Grid1.fit(x_data[['horsepower', 'curb-weight', 'engine-size', 'highway-mpg']], y_data)

Об’єкт знаходить найкращі значення параметрів для даних перевірки. Можемо отримати модель з найкращими параметрами та призначити її змінній BestRR наступним чином:

In [ ]:
BestRR=Grid1.best_estimator_
BestRR

Тепер протестуємо нашу модель на тестових даних:


In [ ]:
BestRR.score(x_test[['horsepower', 'curb-weight', 'engine-size', 'highway-mpg']], y_test)

<div class="alert alert-success alertsuccess" style="margin-top: 20px">
<h2> Завдання  #6: </h2>

Виконайте пошук по сітці для параметра alpha та параметра нормалізації, а потім побудуйте модель, використавши найкращі значення параметрів:
</div>


In [ ]:
# Напишіть ваш код нижче та натисніть Shift+Enter для виконання


<details><summary>Натисніть тут, щоб побачити підказку</summary>

```python
parameters2= [{'alpha': [0.001,0.1,1, 10, 100, 1000,10000,100000,100000],'normalize':[True,False]} ]
Grid2 = GridSearchCV(Ridge(), parameters2,cv=4)
Grid2.fit(x_data[['horsepower', 'curb-weight', 'engine-size', 'highway-mpg']],y_data)
Grid2.best_estimator_


```

</details>



*В теоретичній частині роботи використано елементи курсу "Data Analysis with Python" від IBM Corporation, автор
<a href="https://www.linkedin.com/in/joseph-s-50398b136/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDeveloperSkillsNetworkDA0101ENSkillsNetwork971-2022-01-01" target="_blank">Joseph Santarcangelo</a>*